# RSNA Knee — official-data image baseline training

Private, Internet-disabled Kaggle T4 runner for operation ROOT-RSNA-EXISTING-MODEL-RUNNER-20261015-01.

This notebook embeds the reviewed competition-data-only sources, trains from random initialization, reserves time for held-out evaluation and receipt generation, and does not submit predictions. Outputs remain private until the authorized operator reviews the returned Kaggle version and receipts.


In [ ]:
from pathlib import Path
import base64
import hashlib
import json

SOURCE_B64 = {'train_rsna.py': '...', 'rsna_data.py': '...', 'rsna_image_model.py': 'IiIiQ29tcGV0aXRpb24tZGF0YS1vbmx5LCBpbWFnZS1vbmx5IGhpZXJhcmNoaWNhbCBNSUwgZm9yIFJTTkEgS25lZSAyMDI2LgoKVGhpcyBtb2R1bGUgZG9lcyBub3QgZG93bmxvYWQgZGF0YSBvciB3ZWlnaHRzLiBJdCBpcyBhIHRyYWluYWJsZSBtb2RlbCwgbm90IGEKcHJldHJhaW5lZCBtb2RlbDogdXNlZnVsIHByZWRpY3Rpb25zIHJlcXVpcmUgYW4gYWN0dWFsbHkgdHJhaW5lZCBjaGVja3BvaW50LgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmZyb20gZGF0YWNsYXNzZXMgaW1wb3J0IGFzZGljdCwgZGF0YWNsYXNzCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApmcm9tIHR5cGluZyBpbXBvcnQgTWFwcGluZwoKaW1wb3J0IHRvcmNoCmZyb20gdG9yY2ggaW1wb3J0IFRlbnNvciwgbm4KZnJvbSB0b3JjaC5ubiBpbXBvcnQgZnVuY3Rpb25hbCBhcyBGCmZyb20gdG9yY2gudXRpbHMuY2hlY2twb2ludCBpbXBvcnQgY2hlY2twb2ludAoKCkxBQkVMUyA9ICgKICAgICJBQ0wiLCAiTUNMIiwgIk1lZGlhbCBNZW5pc2N1cyIsICJMYXRlcmFsIE1lbmlzY3VzIiwgIk1lZGlhbCBPQSIsCiAgICAiTGF0ZXJhbCBPQSIsICJQRiBPQSIsICJFZmZ1c2lvbiIsICJTeW5vdml0aXMiLCAiQmFrZXIncyIsCiAgICAiQ29udHVzaW9uIiwgIkZyYWN0dXJlIiwKKQpGT1JNQVRfVkVSU0lPTiA9IDEKCgpAZGF0YWNsYXNzKGZyb3plbj1UcnVlKQpjbGFzcyBNb2RlbENvbmZpZzoKICAgIHdpZHRoOiBpbnQgPSAzMgogICAgZmVhdHVyZV9kaW06IGludCA9IDI1NgogICAgYXR0ZW50aW9uX2RpbTogaW50ID0gMTI4CiAgICBkcm9wb3V0OiBmbG9hdCA9IDAuMgogICAgZW5jb2Rlcl9jaHVua19zaXplOiBpbnQgPSAzMgogICAgZ3JhZGllbnRfY2hlY2twb2ludGluZzogYm9vbCA9IFRydWUKCiAgICBkZWYgX19wb3N0X2luaXRfXyhzZWxmKToKICAgICAgICBpZiBtaW4oc2VsZi53aWR0aCwgc2VsZi5mZWF0dXJlX2RpbSwgc2VsZi5hdHRlbnRpb25fZGltLAogICAgICAgICAgICAgICBzZWxmLmVuY29kZXJfY2h1bmtfc2l6ZSkgPCAxOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJtb2RlbCBkaW1lbnNpb25zIGFuZCBlbmNvZGVyX2NodW5rX3NpemUgbXVzdCBiZSBwb3NpdGl2ZSIpCiAgICAgICAgaWYgbm90IDAgPD0gc2VsZi5kcm9wb3V0IDwgMToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiZHJvcG91dCBtdXN0IGJlIGluIFswLDEpIikKCgpkZWYgX25vcm1hbGl6YXRpb24oY2hhbm5lbHM6IGludCkgLT4gbm4uR3JvdXBOb3JtOgogICAgZ3JvdXBzID0gbWluKDgsIGNoYW5uZWxzKQogICAgd2hpbGUgY2hhbm5lbHMgJSBncm91cHM6CiAgICAgICAgZ3JvdXBzIC09IDEKICAgIHJldHVybiBubi5Hcm91cE5vcm0oZ3JvdXBzLCBjaGFubmVscykKCgpjbGFzcyBSZXNpZHVhbEJsb2NrKG5uLk1vZHVsZSk6CiAgICBkZWYgX19pbml0X18oc2VsZiwgaW5fY2hhbm5lbHM6IGludCwgb3V0X2NoYW5uZWxzOiBpbnQsIHN0cmlkZTogaW50ID0gMSk6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgc2VsZi5ib2R5ID0gbm4uU2VxdWVudGlhbCgKICAgICAgICAgICAgbm4uQ29udjJkKGluX2NoYW5uZWxzLCBvdXRfY2hhbm5lbHMsIDMsIHN0cmlkZSwgMSwgYmlhcz1GYWxzZSksCiAgICAgICAgICAgIF9ub3JtYWxpemF0aW9uKG91dF9jaGFubmVscyksIG5uLlNpTFUoaW5wbGFjZT1GYWxzZSksCiAgICAgICAgICAgIG5uLkNvbnYyZChvdXRfY2hhbm5lbHMsIG91dF9jaGFubmVscywgMywgMSwgMSwgYmlhcz1GYWxzZSksCiAgICAgICAgICAgIF9ub3JtYWxpemF0aW9uKG91dF9jaGFubmVscyksCiAgICAgICAgKQogICAgICAgIHNlbGYuc2hvcnRjdXQgPSAoCiAgICAgICAgICAgIG5uLklkZW50aXR5KCkgaWYgc3RyaWRlID09IDEgYW5kIGluX2NoYW5uZWxzID09IG91dF9jaGFubmVscyBlbHNlCiAgICAgICAgICAgIG5uLlNlcXVlbnRpYWwobm4uQ29udjJkKGluX2NoYW5uZWxzLCBvdXRfY2hhbm5lbHMsIDEsIHN0cmlkZSwgYmlhcz1GYWxzZSksCiAgICAgICAgICAgICAgICAgICAgICAgICAgX25vcm1hbGl6YXRpb24ob3V0X2NoYW5uZWxzKSkKICAgICAgICApCgogICAgZGVmIGZvcndhcmQoc2VsZiwgaW1hZ2VzOiBUZW5zb3IpIC0+IFRlbnNvcjoKICAgICAgICByZXR1cm4gRi5zaWx1KHNlbGYuYm9keShpbWFnZXMpICsgc2VsZi5zaG9ydGN1dChpbWFnZXMpKQoKCmNsYXNzIFNsaWNlRW5jb2Rlcihubi5Nb2R1bGUpOgogICAgIiIiU2luZ2xlLWNoYW5uZWwgcmVzaWR1YWwgQ05OOyBhbGwgcGFyYW1ldGVycyBpbml0aWFsaXplZCBmcm9tIHNjcmF0Y2guIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGNvbmZpZzogTW9kZWxDb25maWcpOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHcgPSBjb25maWcud2lkdGgKICAgICAgICBzZWxmLm5ldHdvcmsgPSBubi5TZXF1ZW50aWFsKAogICAgICAgICAgICBubi5Db252MmQoMSwgdywgNywgMiwgMywgYmlhcz1GYWxzZSksIF9ub3JtYWxpemF0aW9uKHcpLAogICAgICAgICAgICBubi5TaUxVKGlucGxhY2U9RmFsc2UpLCBubi5NYXhQb29sMmQoMywgMiwgMSksCiAgICAgICAgICAgIFJlc2lkdWFsQmxvY2sodywgdyksIFJlc2lkdWFsQmxvY2sodywgdyksCiAgICAgICAgICAgIFJlc2lkdWFsQmxvY2sodywgMiAqIHcsIDIpLCBSZXNpZHVhbEJsb2NrKDIgKiB3LCAyICogdyksCiAgICAgICAgICAgIFJlc2lkdWFsQmxvY2soMiAqIHcsIDQgKiB3LCAyKSwgUmVzaWR1YWxCbG9jayg0ICogdywgNCAqIHcpLAogICAgICAgICAgICBSZXNpZHVhbEJsb2NrKDQgKiB3LCA4ICogdywgMiksIFJlc2lkdWFsQmxvY2soOCAqIHcsIDggKiB3KSwKICAgICAgICAgICAgbm4uQWRhcHRpdmVBdmdQb29sMmQoMSksIG5uLkZsYXR0ZW4oKSwKICAgICAgICAgICAgbm4uTGluZWFyKDggKiB3LCBjb25maWcuZmVhdHVyZV9kaW0pLCBubi5MYXllck5vcm0oY29uZmlnLmZlYXR1cmVfZGltKSwKICAgICAgICApCgogICAgZGVmIGZvcndhcmQoc2VsZiwgaW1hZ2VzOiBUZW5zb3IpIC0+IFRlbnNvcjoKICAgICAgICByZXR1cm4gc2VsZi5uZXR3b3JrKGltYWdlcykKCgpkZWYgbWFza2VkX3NvZnRtYXgobG9naXRzOiBUZW5zb3IsIG1hc2s6IFRlbnNvciwgZGltOiBpbnQpIC0+IFRlbnNvcjoKICAgICIiIkVtcHR5IHBhZGRlZCBiYWdzIHJlY2VpdmUgZXhhY3RseSB6ZXJvIGF0dGVudGlvbiwgbmV2ZXIgTmFOcy4iIiIKICAgIG1hc2sgPSBtYXNrLnRvKGR0eXBlPXRvcmNoLmJvb2wpCiAgICBsb2dpdHMgPSBsb2dpdHMuZmxvYXQoKS5tYXNrZWRfZmlsbCh+bWFzaywgdG9yY2guZmluZm8odG9yY2guZmxvYXQzMikubWluKQogICAgd2VpZ2h0cyA9IHRvcmNoLnNvZnRtYXgobG9naXRzLCBkaW09ZGltKSAqIG1hc2sudG8odG9yY2guZmxvYXQzMikKICAgIHJldHVybiB3ZWlnaHRzIC8gd2VpZ2h0cy5zdW0oZGltPWRpbSwga2VlcGRpbT1UcnVlKS5jbGFtcF9taW4oMWUtMTIpCgoKY2xhc3MgS25lZUltYWdlTUlMKG5uLk1vZHVsZSk6CiAgICAiIiJGaW5kaW5nLXNwZWNpZmljIGF0dGVudGlvbiBvdmVyIHNsaWNlcywgdGhlbiBvdmVyIE1SSSBzZXJpZXMuCgogICAgSW5wdXQgaW1hZ2VzIGFyZSBbYmF0Y2gsc2VyaWVzLHNsaWNlLDEsaGVpZ2h0LHdpZHRoXS4gRElDT00gcGxhbmUvc2VxdWVuY2UKICAgIG1ldGFkYXRhIGlzIFtiYXRjaCxzZXJpZXMsNV07IG5vIHJlcG9ydCwgdGFyZ2V0LCBVSUQgb3IgcGF0aWVudCBpbmZvcm1hdGlvbgogICAgZW50ZXJzIHRoZSBtb2RlbC4gUGFkZGVkIHNsaWNlcyBhcmUgbmV2ZXIgcGFzc2VkIHRocm91Z2ggdGhlIGVuY29kZXIuCiAgICAiIiIKCiAgICBkZWYgX19pbml0X18oc2VsZiwgY29uZmlnOiBNb2RlbENvbmZpZyB8IE5vbmUgPSBOb25lKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLmNvbmZpZyA9IGNvbmZpZyBvciBNb2RlbENvbmZpZygpCiAgICAgICAgZiwgYSA9IHNlbGYuY29uZmlnLmZlYXR1cmVfZGltLCBzZWxmLmNvbmZpZy5hdHRlbnRpb25fZGltCiAgICAgICAgc2VsZi5lbmNvZGVyID0gU2xpY2VFbmNvZGVyKHNlbGYuY29uZmlnKQogICAgICAgIHNlbGYucG9zaXRpb24gPSBubi5TZXF1ZW50aWFsKG5uLkxpbmVhcig0LCBmKSwgbm4uVGFuaCgpKQogICAgICAgIHNlbGYubWV0YWRhdGFfZW1iZWRkaW5nID0gbm4uU2VxdWVudGlhbChubi5MaW5lYXIoNSwgZiksIG5uLlRhbmgoKSkKICAgICAgICBzZWxmLnNsaWNlX3RhbmggPSBubi5MaW5lYXIoZiwgYSkKICAgICAgICBzZWxmLnNsaWNlX2dhdGUgPSBubi5MaW5lYXIoZiwgYSkKICAgICAgICBzZWxmLnNsaWNlX3Njb3JlID0gbm4uTGluZWFyKGEsIGxlbihMQUJFTFMpLCBiaWFzPUZhbHNlKQogICAgICAgIHNlbGYuc2VyaWVzX3RhbmggPSBubi5MaW5lYXIoZiwgYSkKICAgICAgICBzZWxmLnNlcmllc19nYXRlID0gbm4uTGluZWFyKGYsIGEpCiAgICAgICAgc2VsZi5zZXJpZXNfc2NvcmUgPSBubi5QYXJhbWV0ZXIodG9yY2guZW1wdHkobGVuKExBQkVMUyksIGEpKQogICAgICAgIHNlbGYuY2xhc3NpZmllcl93ZWlnaHQgPSBubi5QYXJhbWV0ZXIodG9yY2guZW1wdHkobGVuKExBQkVMUyksIGYpKQogICAgICAgIHNlbGYuY2xhc3NpZmllcl9iaWFzID0gbm4uUGFyYW1ldGVyKHRvcmNoLnplcm9zKGxlbihMQUJFTFMpKSkKICAgICAgICBzZWxmLmRyb3BvdXQgPSBubi5Ecm9wb3V0KHNlbGYuY29uZmlnLmRyb3BvdXQpCiAgICAgICAgbm4uaW5pdC54YXZpZXJfdW5pZm9ybV8oc2VsZi5zZXJpZXNfc2NvcmUpCiAgICAgICAgbm4uaW5pdC54YXZpZXJfdW5pZm9ybV8oc2VsZi5jbGFzc2lmaWVyX3dlaWdodCkKCiAgICBkZWYgX2VuY29kZShzZWxmLCBpbWFnZXM6IFRlbnNvcikgLT4gVGVuc29yOgogICAgICAgIGVuY29kZWQgPSBbXQogICAgICAgIGZvciBjaHVuayBpbiBpbWFnZXMuc3BsaXQoc2VsZi5jb25maWcuZW5jb2Rlcl9jaHVua19zaXplKToKICAgICAgICAgICAgaWYgc2VsZi50cmFpbmluZyBhbmQgc2VsZi5jb25maWcuZ3JhZGllbnRfY2hlY2twb2ludGluZzoKICAgICAgICAgICAgICAgIGVuY29kZWQuYXBwZW5kKGNoZWNrcG9pbnQoc2VsZi5lbmNvZGVyLCBjaHVuaywgdXNlX3JlZW50cmFudD1GYWxzZSkpCiAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICBlbmNvZGVkLmFwcGVuZChzZWxmLmVuY29kZXIoY2h1bmspKQogICAgICAgIHJldHVybiB0b3JjaC5jYXQoZW5jb2RlZCwgZGltPTApCgogICAgZGVmIGZvcndhcmQoc2VsZiwgYmF0Y2g6IE1hcHBpbmdbc3RyLCBUZW5zb3JdKSAtPiBUZW5zb3I6CiAgICAgICAgaW1hZ2VzID0gYmF0Y2hbImltYWdlcyJdCiAgICAgICAgc2xpY2VfbWFzayA9IGJhdGNoWyJzbGljZV9tYXNrIl0uYm9vbCgpCiAgICAgICAgc2VyaWVzX21hc2sgPSBiYXRjaFsic2VyaWVzX21hc2siXS5ib29sKCkKICAgICAgICBtZXRhZGF0YSA9IGJhdGNoWyJtZXRhZGF0YSJdCiAgICAgICAgaWYgaW1hZ2VzLm5kaW0gIT0gNiBvciBpbWFnZXMuc2hhcGVbM10gIT0gMToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiaW1hZ2VzIG11c3QgYmUgW0IsUyxELDEsSCxXXSIpCiAgICAgICAgYiwgcywgZCA9IGltYWdlcy5zaGFwZVs6M10KICAgICAgICBpZiBzbGljZV9tYXNrLnNoYXBlICE9IChiLCBzLCBkKSBvciBzZXJpZXNfbWFzay5zaGFwZSAhPSAoYiwgcyk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInNsaWNlL3NlcmllcyBtYXNrIHNoYXBlIG1pc21hdGNoIikKICAgICAgICBpZiBtZXRhZGF0YS5zaGFwZSAhPSAoYiwgcywgNSk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIm1ldGFkYXRhIG11c3QgYmUgW0IsUyw1XSIpCiAgICAgICAgYWN0aXZlID0gc2xpY2VfbWFzayAmIHNlcmllc19tYXNrLnVuc3F1ZWV6ZSgtMSkKICAgICAgICBhY3R1YWxfc2VyaWVzID0gYWN0aXZlLmFueShkaW09LTEpCiAgICAgICAgaWYgbm90IGFjdHVhbF9zZXJpZXMuYW55KGRpbT0tMSkuYWxsKCk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImVhY2ggc3R1ZHkgbXVzdCBjb250YWluIGF0IGxlYXN0IG9uZSBkZWNvZGVkIGltYWdlIikKICAgICAgICBpZiBub3QgdG9yY2guaXNmaW5pdGUobWV0YWRhdGEpLmFsbCgpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJub24tZmluaXRlIE1SSSBtZXRhZGF0YSIpCiAgICAgICAgZmxhdF9pbWFnZXMgPSBpbWFnZXMucmVzaGFwZShiICogcyAqIGQsICppbWFnZXMuc2hhcGVbMzpdKQogICAgICAgIHZhbGlkX2luZGV4ID0gYWN0aXZlLnJlc2hhcGUoLTEpLm5vbnplcm8oYXNfdHVwbGU9RmFsc2UpLmZsYXR0ZW4oKQogICAgICAgIHNlbGVjdGVkID0gZmxhdF9pbWFnZXMuaW5kZXhfc2VsZWN0KDAsIHZhbGlkX2luZGV4KQogICAgICAgIGlmIG5vdCB0b3JjaC5pc2Zpbml0ZShzZWxlY3RlZCkuYWxsKCk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIm5vbi1maW5pdGUgZGVjb2RlZCBNUkkgcGl4ZWxzIikKICAgICAgICB2YWxpZF9mZWF0dXJlcyA9IHNlbGYuX2VuY29kZShzZWxlY3RlZCkKICAgICAgICBmZWF0dXJlcyA9IHZhbGlkX2ZlYXR1cmVzLm5ld196ZXJvcygoYiAqIHMgKiBkLCBzZWxmLmNvbmZpZy5mZWF0dXJlX2RpbSkpCiAgICAgICAgZmVhdHVyZXMgPSBmZWF0dXJlcy5pbmRleF9jb3B5KDAsIHZhbGlkX2luZGV4LCB2YWxpZF9mZWF0dXJlcykudmlldyhiLCBzLCBkLCAtMSkKICAgICAgICAjIFJlbGF0aXZlIHNsaWNlIG9yZGVyLCBub3Qgc3R1ZHkgaWRlbnRpdHk7IHBhZGRlZCBzZXJpZXMgcmVtYWluIG1hc2tlZC4KICAgICAgICByYW5rID0gYWN0aXZlLmN1bXN1bShkaW09LTEpLnRvKGZlYXR1cmVzLmR0eXBlKSAtIDEKICAgICAgICBkZW5vbSA9IChhY3RpdmUuc3VtKGRpbT0tMSwga2VlcGRpbT1UcnVlKSAtIDEpLmNsYW1wX21pbigxKQogICAgICAgIHBvc2l0aW9uID0gMiAqIHJhbmsgLyBkZW5vbSAtIDEKICAgICAgICBwb3NpdGlvbmFsID0gdG9yY2guc3RhY2soKHBvc2l0aW9uLCBwb3NpdGlvbi5zcXVhcmUoKSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHRvcmNoLnNpbih0b3JjaC5waSAqIHBvc2l0aW9uKSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHRvcmNoLmNvcyh0b3JjaC5waSAqIHBvc2l0aW9uKSksIGRpbT0tMSkKICAgICAgICBmZWF0dXJlcyA9IGZlYXR1cmVzICsgc2VsZi5wb3NpdGlvbihwb3NpdGlvbmFsKQogICAgICAgIGdhdGVkID0gdG9yY2gudGFuaChzZWxmLnNsaWNlX3RhbmgoZmVhdHVyZXMpKSAqIHRvcmNoLnNpZ21vaWQoc2VsZi5zbGljZV9nYXRlKGZlYXR1cmVzKSkKICAgICAgICBhdHRlbnRpb24gPSBtYXNrZWRfc29mdG1heChzZWxmLnNsaWNlX3Njb3JlKGdhdGVkKSwgYWN0aXZlLnVuc3F1ZWV6ZSgtMSksIGRpbT0yKQogICAgICAgICMgW0IsUyxMLEZdOiBlYWNoIGZpbmRpbmcgYXR0ZW5kcyB0byBkaWZmZXJlbnQgZXZpZGVuY2Ugc2xpY2VzLgogICAgICAgIHNlcmllc19mZWF0dXJlcyA9IHRvcmNoLmVpbnN1bSgiYnNkbCxic2RmLT5ic2xmIiwgYXR0ZW50aW9uLnRvKGZlYXR1cmVzLmR0eXBlKSwgZmVhdHVyZXMpCiAgICAgICAgc2VyaWVzX2ZlYXR1cmVzID0gc2VyaWVzX2ZlYXR1cmVzICsgc2VsZi5tZXRhZGF0YV9lbWJlZGRpbmcobWV0YWRhdGEudG8oZmVhdHVyZXMuZHR5cGUpKS51bnNxdWVlemUoMikKICAgICAgICBnYXRlZF9zZXJpZXMgPSB0b3JjaC50YW5oKHNlbGYuc2VyaWVzX3Rhbmgoc2VyaWVzX2ZlYXR1cmVzKSkgKiB0b3JjaC5zaWdtb2lkKHNlbGYuc2VyaWVzX2dhdGUoc2VyaWVzX2ZlYXR1cmVzKSkKICAgICAgICBzZXJpZXNfbG9naXRzID0gdG9yY2guZWluc3VtKCJic2xhLGxhLT5ic2wiLCBnYXRlZF9zZXJpZXMsIHNlbGYuc2VyaWVzX3Njb3JlKQogICAgICAgIHNlcmllc19hdHRlbnRpb24gPSBtYXNrZWRfc29mdG1heChzZXJpZXNfbG9naXRzLCBhY3R1YWxfc2VyaWVzLnVuc3F1ZWV6ZSgtMSksIGRpbT0xKQogICAgICAgIHN0dWR5X2ZlYXR1cmVzID0gdG9yY2guZWluc3VtKCJic2wsYnNsZi0+YmxmIiwgc2VyaWVzX2F0dGVudGlvbi50byhmZWF0dXJlcy5kdHlwZSksIHNlcmllc19mZWF0dXJlcykKICAgICAgICByZXR1cm4gdG9yY2guZWluc3VtKCJibGYsbGYtPmJsIiwgc2VsZi5kcm9wb3V0KHN0dWR5X2ZlYXR1cmVzKSwgc2VsZi5jbGFzc2lmaWVyX3dlaWdodCkgKyBzZWxmLmNsYXNzaWZpZXJfYmlhcwoKCmRlZiBtYXNrZWRfYmNlX2xvc3MobG9naXRzOiBUZW5zb3IsIHRhcmdldHM6IFRlbnNvciwgbWFzazogVGVuc29yLAogICAgICAgICAgICAgICAgICAgIHBvc193ZWlnaHQ6IFRlbnNvciB8IE5vbmUgPSBOb25lLAogICAgICAgICAgICAgICAgICAgIGNvbmZpZGVuY2U6IFRlbnNvciB8IE5vbmUgPSBOb25lKSAtPiBUZW5zb3I6CiAgICAiIiJNYWNyby1sYWJlbCBCQ0U6IG1pc3NpbmcgdGFyZ2V0cyBoYXZlIHplcm8gbG9zcyBhbmQgemVybyBncmFkaWVudC4KCiAgICBPcHRpb25hbCBjb25maWRlbmNlIHdlaWdodHMgc3VwcG9ydCBleHBsaWNpdGx5IGFkbWl0dGVkIHdlYWsgdHJhaW5pbmcgbGFiZWxzOwogICAgdmFsaWRhdGlvbiBtdXN0IHVzZSB0aGUgc2VwYXJhdGUgZ29sZCBtYXNrLiBQb3NpdGl2ZSB3ZWlnaHRzIGNvbWUgZnJvbSB0aGUKICAgIHRyYWluaW5nIGZvbGQgb25seS4gR29sZCB0YXJnZXRzIGFyZSBiaW5hcnk7IGFkbWl0dGVkIHdlYWsgdGFyZ2V0cyBtYXkgYmUKICAgIHNvZnQgcHJvYmFiaWxpdGllcyBpbiBbMCwxXS4KICAgICIiIgogICAgaWYgbG9naXRzLnNoYXBlICE9IHRhcmdldHMuc2hhcGUgb3IgbG9naXRzLnNoYXBlICE9IG1hc2suc2hhcGU6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigibG9naXRzLCB0YXJnZXRzIGFuZCBtYXNrIG11c3QgaGF2ZSBpZGVudGljYWwgW0IsMTJdIHNoYXBlIikKICAgIG9ic2VydmVkID0gbWFzay5ib29sKCkKICAgIGlmIG5vdCBvYnNlcnZlZC5hbnkoKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJiYXRjaCBoYXMgbm8gb2JzZXJ2ZWQgdGFyZ2V0cyIpCiAgICBpZiBub3QgdG9yY2guaXNmaW5pdGUodGFyZ2V0c1tvYnNlcnZlZF0pLmFsbCgpIG9yIG5vdCAoKHRhcmdldHNbb2JzZXJ2ZWRdID49IDApICYgKHRhcmdldHNbb2JzZXJ2ZWRdIDw9IDEpKS5hbGwoKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJvYnNlcnZlZCB0YXJnZXRzIG11c3QgYmUgZmluaXRlIHByb2JhYmlsaXRpZXMgaW4gWzAsMV0iKQogICAgc2FmZV90YXJnZXRzID0gdG9yY2gud2hlcmUob2JzZXJ2ZWQsIHRhcmdldHMsIHRvcmNoLnplcm9zX2xpa2UodGFyZ2V0cykpLmZsb2F0KCkKICAgIHdlaWdodHMgPSBvYnNlcnZlZC5mbG9hdCgpCiAgICBpZiBjb25maWRlbmNlIGlzIG5vdCBOb25lOgogICAgICAgIGlmIGNvbmZpZGVuY2Uuc2hhcGUgIT0gbG9naXRzLnNoYXBlIG9yIG5vdCB0b3JjaC5pc2Zpbml0ZShjb25maWRlbmNlKS5hbGwoKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiaW52YWxpZCB0YXJnZXQgY29uZmlkZW5jZSBzaGFwZS92YWx1ZSIpCiAgICAgICAgaWYgbm90ICgoY29uZmlkZW5jZSA+PSAwKSAmIChjb25maWRlbmNlIDw9IDEpKS5hbGwoKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigidGFyZ2V0IGNvbmZpZGVuY2UgbXVzdCBiZSBpbiBbMCwxXSIpCiAgICAgICAgd2VpZ2h0cyA9IHdlaWdodHMgKiBjb25maWRlbmNlLmZsb2F0KCkKICAgIHJhdyA9IEYuYmluYXJ5X2Nyb3NzX2VudHJvcHlfd2l0aF9sb2dpdHMobG9naXRzLmZsb2F0KCksIHNhZmVfdGFyZ2V0cywKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgcG9zX3dlaWdodD1wb3Nfd2VpZ2h0LCByZWR1Y3Rpb249Im5vbmUiKQogICAgZGVub21pbmF0b3JzID0gd2VpZ2h0cy5zdW0oZGltPTApCiAgICBwcmVzZW50ID0gZGVub21pbmF0b3JzID4gMAogICAgaWYgbm90IHByZXNlbnQuYW55KCk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiYmF0Y2ggaGFzIG5vIHBvc2l0aXZlLWNvbmZpZGVuY2Ugb2JzZXJ2ZWQgdGFyZ2V0cyIpCiAgICByZXR1cm4gKChyYXcgKiB3ZWlnaHRzKS5zdW0oZGltPTApW3ByZXNlbnRdIC8gZGVub21pbmF0b3JzW3ByZXNlbnRdKS5tZWFuKCkKCgpkZWYgbW92ZV9tb2RlbF9pbnB1dHMoYmF0Y2g6IE1hcHBpbmcsIGRldmljZTogc3RyIHwgdG9yY2guZGV2aWNlKSAtPiBkaWN0W3N0ciwgVGVuc29yXToKICAgIHJldHVybiB7a2V5OiBiYXRjaFtrZXldLnRvKGRldmljZSwgbm9uX2Jsb2NraW5nPVRydWUpCiAgICAgICAgICAgIGZvciBrZXkgaW4gKCJpbWFnZXMiLCAic2xpY2VfbWFzayIsICJzZXJpZXNfbWFzayIsICJtZXRhZGF0YSIpfQoKCkB0b3JjaC5pbmZlcmVuY2VfbW9kZSgpCmRlZiBwcmVkaWN0X2JhdGNoKG1vZGVsOiBLbmVlSW1hZ2VNSUwsIGJhdGNoOiBNYXBwaW5nLAogICAgICAgICAgICAgICAgICBkZXZpY2U6IHN0ciB8IHRvcmNoLmRldmljZSA9ICJjcHUiKSAtPiBUZW5zb3I6CiAgICBtb2RlbC5ldmFsKCkKICAgIHByb2JhYmlsaXRpZXMgPSB0b3JjaC5zaWdtb2lkKG1vZGVsKG1vdmVfbW9kZWxfaW5wdXRzKGJhdGNoLCBkZXZpY2UpKS5mbG9hdCgpKQogICAgaWYgbm90IHRvcmNoLmlzZmluaXRlKHByb2JhYmlsaXRpZXMpLmFsbCgpOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigibW9kZWwgcHJvZHVjZWQgbm9uLWZpbml0ZSBwcm9iYWJpbGl0aWVzIikKICAgIHJldHVybiBwcm9iYWJpbGl0aWVzCgoKZGVmIGxvYWRfbW9kZWwoY2hlY2twb2ludF9wYXRoOiBzdHIgfCBQYXRoLCBkZXZpY2U6IHN0ciB8IHRvcmNoLmRldmljZSA9ICJjcHUiKToKICAgICIiIkxvYWQgb3VyIHN0YXRlLWRpY3Qtb25seSBjaGVja3BvaW50IHdpdGhvdXQgZW5hYmxpbmcgYXJiaXRyYXJ5IHVucGlja2xpbmcuIiIiCiAgICBzYXZlZCA9IHRvcmNoLmxvYWQoUGF0aChjaGVja3BvaW50X3BhdGgpLCBtYXBfbG9jYXRpb249ImNwdSIsIHdlaWdodHNfb25seT1UcnVlKQogICAgaWYgc2F2ZWQuZ2V0KCJmb3JtYXRfdmVyc2lvbiIpICE9IEZPUk1BVF9WRVJTSU9OIG9yIHR1cGxlKHNhdmVkLmdldCgibGFiZWxzIiwgKCkpKSAhPSBMQUJFTFM6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiY2hlY2twb2ludCBmb3JtYXQgb3IgMTItbGFiZWwgb3JkZXIgZG9lcyBub3QgbWF0Y2ggdGhpcyBtb2RlbCIpCiAgICBpZiBzYXZlZC5nZXQoImFyY2hpdGVjdHVyZSIpICE9ICJLbmVlSW1hZ2VNSUwiOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInVuc3VwcG9ydGVkIGNoZWNrcG9pbnQgYXJjaGl0ZWN0dXJlIikKICAgIG1vZGVsID0gS25lZUltYWdlTUlMKE1vZGVsQ29uZmlnKCoqc2F2ZWRbIm1vZGVsX2NvbmZpZyJdKSkKICAgIG1vZGVsLmxvYWRfc3RhdGVfZGljdChzYXZlZFsibW9kZWxfc3RhdGUiXSwgc3RyaWN0PVRydWUpCiAgICBtb2RlbC50byhkZXZpY2UpLmV2YWwoKQogICAgbWV0YWRhdGEgPSB7a2V5OiB2YWx1ZSBmb3Iga2V5LCB2YWx1ZSBpbiBzYXZlZC5pdGVtcygpCiAgICAgICAgICAgICAgICBpZiBrZXkgbm90IGluIHsibW9kZWxfc3RhdGUiLCAib3B0aW1pemVyX3N0YXRlIiwgInNjaGVkdWxlcl9zdGF0ZSIsICJzY2FsZXJfc3RhdGUifX0KICAgIHJldHVybiBtb2RlbCwgbWV0YWRhdGEKCgpkZWYgY2hlY2twb2ludF9wYXlsb2FkKG1vZGVsOiBLbmVlSW1hZ2VNSUwsICoqbWV0YWRhdGEpIC0+IGRpY3Q6CiAgICByZXR1cm4geyoqbWV0YWRhdGEsICJmb3JtYXRfdmVyc2lvbiI6IEZPUk1BVF9WRVJTSU9OLCAiYXJjaGl0ZWN0dXJlIjogIktuZWVJbWFnZU1JTCIsCiAgICAgICAgICAgICJtb2RlbF9jb25maWciOiBhc2RpY3QobW9kZWwuY29uZmlnKSwgImxhYmVscyI6IGxpc3QoTEFCRUxTKSwKICAgICAgICAgICAgIm1vZGVsX3N0YXRlIjogbW9kZWwuc3RhdGVfZGljdCgpfQo=', 'rsna_metrics.py': '...'}
SOURCE_SHA256 = {'train_rsna.py': '85641dd9f628e40c312621e1f7689c3221f48b760954e833690835671857a164', 'rsna_data.py': 'a7e14e03703e42933a92c2e17e89cc0236d785e8c43797226329219347175e03', 'rsna_image_model.py': '0e694bbc533e8058bfc35f60c6d0bc3d59f34c3ea05e13dd641fadc64c9c46fa', 'rsna_metrics.py': 'c73a18067c3d5bef421135d518f02bd6b6eff33189398fc3577bf51deda4e74f'}
SOURCE_DIR = Path("/kaggle/working/rsna_source")
SOURCE_DIR.mkdir(parents=True, exist_ok=True)

for filename, payload in SOURCE_B64.items():
    content = base64.b64decode(payload, validate=True)
    actual = hashlib.sha256(content).hexdigest()
    if actual != SOURCE_SHA256[filename]:
        raise RuntimeError(f"embedded source integrity failure: {filename}")
    (SOURCE_DIR / filename).write_bytes(content)

print(json.dumps({
    "source_materialized": True,
    "files": sorted(SOURCE_SHA256),
    "source_sha256": SOURCE_SHA256,
}, sort_keys=True))


In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import pydicom
import torch
from PIL import Image

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required; this notebook must run on a Kaggle GPU session")
if torch.cuda.device_count() != 1:
    raise RuntimeError(f"expected one assigned GPU, found {torch.cuda.device_count()}")

handlers = []
for handler in getattr(pydicom.config, "pixel_data_handlers", []):
    try:
        if handler.is_available():
            handlers.append(handler.__name__)
    except Exception:
        continue
if not handlers:
    raise RuntimeError("pydicom has no available pixel-data handler")

mount = Path("/kaggle/input")
slug = "rsna-knee-abnormality-detection"
candidates = []
for train_csv in mount.rglob("train.csv"):
    base = train_csv.parent.resolve()
    if slug not in base.parts:
        continue
    if (base / "train_series.csv").is_file() and (base / "train_series").is_dir():
        candidates.append(base)
candidates = sorted(set(candidates))
if len(candidates) != 1:
    raise RuntimeError(f"expected exactly one official {slug} data root, found {len(candidates)}")
DATA_ROOT = candidates[0]

print(json.dumps({
    "environment_verified": True,
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "cuda_device": torch.cuda.get_device_name(0),
    "numpy": np.__version__,
    "pydicom": pydicom.__version__,
    "pillow": Image.__version__,
    "pixel_handlers": handlers,
    "official_root": str(DATA_ROOT),
}, sort_keys=True))


In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import time

OUTPUT_DIR = Path("/kaggle/working/rsna-baseline")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
command = [
    sys.executable, str(SOURCE_DIR / "train_rsna.py"),
    "--data-root", str(DATA_ROOT),
    "--output", str(OUTPUT_DIR),
    "--operation-id", "ROOT-RSNA-EXISTING-MODEL-RUNNER-20261015-01",
    "--official-data-confirmed",
    "--device", "cuda",
    "--epochs", "20",
    "--batch-size", "1",
    "--accumulation-steps", "8",
    "--workers", "2",
    "--max-runtime-seconds", "28800",
    "--validation-reserve-seconds", "1800",
]
started = time.monotonic()
completed = subprocess.run(command, cwd=SOURCE_DIR, check=False)
elapsed = time.monotonic() - started
if completed.returncode != 0:
    raise RuntimeError(f"training process exited {completed.returncode}; inspect aggregate training_log.jsonl")
if elapsed > 30600:
    raise RuntimeError(f"training exceeded the 8.5-hour internal execution boundary: {elapsed:.1f}s")
print(json.dumps({"training_process_complete": True, "elapsed_seconds": elapsed}, sort_keys=True))


In [ ]:
from pathlib import Path
import hashlib
import json

def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

selected_path = OUTPUT_DIR / "selected_model.json"
manifest_path = OUTPUT_DIR / "run_manifest.json"
if not selected_path.is_file() or not manifest_path.is_file():
    raise RuntimeError("required training receipts are absent")
selected = json.loads(selected_path.read_text(encoding="utf-8"))
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
if manifest.get("status") != "COMPLETED":
    raise RuntimeError(f"run manifest is not completed: {manifest.get('status')!r}")
if selected.get("operation_id") != "ROOT-RSNA-EXISTING-MODEL-RUNNER-20261015-01":
    raise RuntimeError("selected checkpoint operation ID mismatch")
checkpoint = OUTPUT_DIR / selected.get("checkpoint_filename", "")
if not checkpoint.is_file():
    raise RuntimeError("selected checkpoint file is absent")
checkpoint_sha = sha256(checkpoint)
if checkpoint_sha != selected.get("checkpoint_sha256"):
    raise RuntimeError("selected checkpoint SHA-256 mismatch")
manifest_best = manifest.get("artifacts", {}).get("best.pt", {}).get("sha256")
if manifest_best != checkpoint_sha:
    raise RuntimeError("run manifest best-checkpoint SHA-256 mismatch")

receipt = {
    "schema_version": 1,
    "operation_id": selected["operation_id"],
    "status": manifest["status"],
    "stop_reason": manifest.get("stop_reason"),
    "optimizer_steps": manifest.get("optimizer_steps"),
    "selected_epoch": selected.get("epoch"),
    "criterion": selected.get("criterion"),
    "criterion_value": selected.get("criterion_value"),
    "validation": selected.get("validation"),
    "checkpoint": {
        "filename": checkpoint.name,
        "sha256": checkpoint_sha,
        "bytes": checkpoint.stat().st_size,
    },
    "source_sha256": selected.get("source_sha256"),
    "evidence_boundary": {
        "official_leaderboard_score": None,
        "submission_id": None,
        "contains_medical_identifiers": False,
    },
}
receipt_path = OUTPUT_DIR / "KAGGLE_TRAINING_RECEIPT.json"
receipt_path.write_text(json.dumps(receipt, indent=2, sort_keys=True) + "\n", encoding="utf-8")
print(json.dumps(receipt, sort_keys=True))
